In [10]:
import pandas as pd
import numpy as np
import os
from evcouplings.utils import read_config_file, write_config_file
import glob

In [2]:
#load config file
df = pd.read_csv('/n/groups/marks/projects/viral_families/priority-viruses/data/reference_files/viral_dms_reference.csv')

In [3]:
df

,DMS ID,Viral Family,Virus,Protein,Author,Title,Year,Assay,Type,In ProteinGym,Sequence
0,LASSA_GP_Carr,Arenaviridae,Lassa,GP,Carr,Deep mutational scanning reveals functional co...,2024,fitness,Eukaryotic virus,No,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
1,PESV_POLG_Tsuboyama,Caliciviridae,Porcine enteric sapovirus,POLG,Tsuboyama,Mega-scale experimental analysis of protein fo...,2023,stability,Eukaryotic virus,Yes,ALRDDEYDEWQDIIRDWRKEMTVQQFLDLKERALSGASDPDSQRYN...
2,SARS2_PLPRO_Wu_abundance,Coronaviridae,SARS-CoV-2,PLPRO,Wu,Mutational profiling of SARS-CoV-2 papain-like...,2024,abundance,Eukaryotic virus,No,MEVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIK...
3,SARS2_PLPRO_Wu_activity,Coronaviridae,SARS-CoV-2,PLPRO,Wu,Mutational profiling of SARS-CoV-2 papain-like...,2024,activity,Eukaryotic virus,No,MEVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIK...
4,SARS2_PRD0038_RBD_Starr,Coronaviridae,Bat coronavirus PRD0038,RBD,Starr,https://github.com/tstarrlab/SARSr-CoV-RBD_DMS,2023,expression,Eukaryotic virus,No,MKFFILLSLLPFATAQEGCGILSNKSKPALTQYSSSRRGFYYFDDT...
5,SARS2_MRPO_Flynn,Coronaviridae,SARS-CoV-2,MRPO,Flynn,Comprehensive fitness landscape of SARS-CoV-2 ...,2022,fitness,Eukaryotic virus,Yes,SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTS...
6,RmYN02_RBD_Starr,Coronaviridae,Bat coronavirus RmYN02,RBD,Starr,https://github.com/tstarrlab/SARSr-CoV-RBD_DMS,2023,expression,Eukaryotic virus,No,MFILLLIGYTAATTCVTGPTTENKQNVSSLMRGVYYPDDIYRSNVN...
7,RsYN04_RBD_Starr,Coronaviridae,Bat coronavirus RsYN04,RBD,Starr,https://github.com/tstarrlab/SARSr-CoV-RBD_DMS,2023,expression,Eukaryotic virus,No,MFILLLLPIVLAQQDSCNHIVQLPNSMVRGVYNSGSKVYYPDDINR...
8,SARS2_XBB15_RBD_Taylor,Coronaviridae,SARS-CoV-2 XBB.1.5,RBD,Taylor,Deep mutational scans of XBB.1.5 and BQ.1.1 re...,2023,expression,Eukaryotic virus,No,MFVFLVLLPLVSSQCVNLITRTQSYTNSFTRGVYYPDKVFRSSVLH...
9,SARS2_BA1_SPIKE_Dadonaite,Coronaviridae,SARS-CoV-2 BA.1,SPIKE,Dadonaite,A pseudovirus system enables deep mutational s...,2023,fitness,Eukaryotic virus,No,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...


In [8]:
#make individual fasta files to run EVCoupling

path = '/n/groups/marks/projects/viral_families/notebooks/navami/evmutation_runs/'

for i, row in df.iterrows():
    seq = row['Sequence']
    vir_prot = row['DMS ID']

    if os.path.exists(path + vir_prot + ".fa"):
        pass 
        print(i, vir_prot, 'folder already exists')
    else:
        with open(path + vir_prot + ".fa", 'w') as out_fa:
            out_fa.write('>'+ vir_prot + '\n' + seq + '\n')
        print(i, vir_prot, '**NEW**')



In [4]:
## Specify parameters 
theta = 0.99
seq_cov = 50 # Only keep sequences that align to at least x% of the target sequence (i.e. remove fragments)
col_cov = 50  # Only include alignment columns with at least x% residues (rather than gaps) during model inference


#select which database to use to build an alignment
#database = 'uniref_bfd_mgnify'
#database = 'uniref100'
database = 'uniref90'

In [11]:
## Make config files

main_config_path = "/n/groups/marks/projects/viral_families/priority-viruses/scripts/EVCoupling_Config_Scripts/config_monomer_"+database+".txt"

files = []
for i, row in df.iterrows():
    # Naming convention for ESCAPE DMSs 
    seq= row['Sequence']
    vir_prot = row['DMS ID']
    seq_path = path + vir_prot + ".fa" 
    out_config_path = path + 'config/' + vir_prot + '_config_'+database+'.txt'
    config = read_config_file(main_config_path, preserve_order=True)
    prefix_full = vir_prot + "_seqcov" + str(seq_cov) + "_colcov" + str(col_cov) + "_theta" + str(theta)

    if os.path.exists(out_config_path):
        pass 
        print(prefix_full, 'folder already exists')
    else:
        config["global"]["prefix"] = path + 'output/' + database + "/" + prefix_full + "/" + prefix_full
        config["global"]["sequence_id"] = vir_prot
        config["global"]["sequence_file"] = seq_path
        config["global"]["theta"] = theta
    
        config["align"]['compute_num_effective_seqs'] = True
        config["align"]['minimum_sequence_coverage']  = seq_cov
        config["align"]['minimum_column_coverage']    = col_cov
    
        write_config_file(out_config_path, config)
        files.append(out_config_path)
        print(prefix_full, '**NEW**')


LASSA_GP_Carr_seqcov50_colcov50_theta0.99 **NEW**
PESV_POLG_Tsuboyama_seqcov50_colcov50_theta0.99 **NEW**
SARS2_PLPRO_Wu_abundance_seqcov50_colcov50_theta0.99 **NEW**
SARS2_PLPRO_Wu_activity_seqcov50_colcov50_theta0.99 **NEW**
SARS2_PRD0038_RBD_Starr_seqcov50_colcov50_theta0.99 **NEW**
SARS2_MRPO_Flynn_seqcov50_colcov50_theta0.99 **NEW**
RmYN02_RBD_Starr_seqcov50_colcov50_theta0.99 **NEW**
RsYN04_RBD_Starr_seqcov50_colcov50_theta0.99 **NEW**
SARS2_XBB15_RBD_Taylor_seqcov50_colcov50_theta0.99 **NEW**
SARS2_BA1_SPIKE_Dadonaite_seqcov50_colcov50_theta0.99 **NEW**
SARS2_DELTA_SPIKE_Dadonaite_seqcov50_colcov50_theta0.99 **NEW**
SARS2_RBD_Starr_binding_seqcov50_colcov50_theta0.99 **NEW**
SARS2_RBD_Starr_expression_seqcov50_colcov50_theta0.99 **NEW**
DENV_POLG_Suphatrakul_seqcov50_colcov50_theta0.99 **NEW**
IAV_H1_HA_Doud_seqcov50_colcov50_theta0.99 **NEW**
IAV_H1_HA_Wu_seqcov50_colcov50_theta0.99 **NEW**
IAV_PB2_Soh_seqcov50_colcov50_theta0.99 **NEW**
IAV_H3_HA_Lee_seqcov50_colcov50_theta0.9

In [12]:
##Make run scripts

file_path = "/n/groups/marks/projects/viral_families/priority-viruses/scripts/run_evcoupling.sh"

files = list(set(files))
with open(file_path, 'w') as file:
    for item in files:
        file.write('evcouplings ' + item + '\n')